# MNIST Denoising Autoencoder

This notebook builds a **Convolutional Denoising Autoencoder** that learns to remove noise from MNIST digit images.

## Architecture Overview
```
Noisy Image → [Encoder] → Latent Space → [Decoder] → Clean Image
```

- **Encoder**: Conv2D layers that compress the image into a compact representation
- **Latent Space**: Bottleneck that forces the model to learn meaningful features
- **Decoder**: Transposed Conv2D layers that reconstruct the clean image

## What We'll Do
1. Load and preprocess MNIST data from the provided PNG files (or download via Keras)
2. Add synthetic Gaussian noise to training images
3. Train the autoencoder to map noisy → clean
4. Evaluate and visualize results

## 1. Install & Import Dependencies

In [1]:
# Install required packages if needed
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Core packages (usually pre-installed in Colab/Jupyter)
try:
    import tensorflow
except ImportError:
    install('tensorflow')

try:
    import matplotlib
except ImportError:
    install('matplotlib')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile
import random

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from PIL import Image

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load MNIST Data

We support two sources:
- **Option A** (default): Load directly from Keras — fast, no extraction needed
- **Option B**: Load from the uploaded `archive.zip` PNG files

Set `USE_ZIP_FILE = True` and `ZIP_PATH` to use the PNG dataset.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
USE_ZIP_FILE = False          # Set to True to load from archive.zip
ZIP_PATH     = 'archive.zip'  # Path to your uploaded zip file
EXTRACT_DIR  = './mnist_png'  # Where to extract PNGs
# ─────────────────────────────────────────────────────────────────

IMG_SHAPE = (28, 28, 1)

def load_from_keras():
    """Fast path: use Keras built-in MNIST."""
    (x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32')  / 255.0
    x_train = x_train[..., np.newaxis]  # (60000, 28, 28, 1)
    x_test  = x_test[..., np.newaxis]   # (10000, 28, 28, 1)
    return x_train, x_test


def load_from_zip(zip_path, extract_dir):
    """Load PNG files from the archive.zip dataset."""
    if not os.path.isdir(extract_dir):
        print(f'Extracting {zip_path} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('.')  # Extracts to mnist_png/
        print('Done.')

    images = []
    # Walk both training/ and testing/ sub-folders, digits 0–9
    for split in ['training', 'testing']:
        split_dir = os.path.join(extract_dir, split)
        if not os.path.isdir(split_dir):
            continue
        for digit in sorted(os.listdir(split_dir)):
            digit_dir = os.path.join(split_dir, digit)
            if not os.path.isdir(digit_dir):
                continue
            for fname in os.listdir(digit_dir):
                if fname.lower().endswith('.png'):
                    img = Image.open(os.path.join(digit_dir, fname)).convert('L')
                    images.append(np.array(img))

    data = np.stack(images, axis=0).astype('float32') / 255.0
    data = data[..., np.newaxis]  # Add channel dim
    np.random.shuffle(data)

    split = int(len(data) * 0.85)
    return data[:split], data[split:]


# ── Load data ────────────────────────────────────────────────────
if USE_ZIP_FILE and os.path.exists(ZIP_PATH):
    print('Loading from archive.zip ...')
    x_train, x_test = load_from_zip(ZIP_PATH, EXTRACT_DIR)
else:
    print('Loading MNIST from Keras ...')
    x_train, x_test = load_from_keras()

print(f'Train shape : {x_train.shape}  |  min={x_train.min():.2f}, max={x_train.max():.2f}')
print(f'Test  shape : {x_test.shape}   |  min={x_test.min():.2f},  max={x_test.max():.2f}')

## 3. Add Gaussian Noise

We corrupt the clean images with **Gaussian noise** at a configurable level. The autoencoder will learn to reconstruct the clean version.

In [ ]:
NOISE_FACTOR = 0.4  # Try values 0.2–0.6 for different difficulty levels

def add_noise(images, noise_factor=NOISE_FACTOR):
    """Add Gaussian noise and clip to [0, 1]."""
    noise   = np.random.normal(loc=0.0, scale=noise_factor, size=images.shape)
    noisy   = images + noise
    noisy   = np.clip(noisy, 0.0, 1.0)
    return noisy.astype('float32')

x_train_noisy = add_noise(x_train)
x_test_noisy  = add_noise(x_test)

print(f'Noisy train range: [{x_train_noisy.min():.3f}, {x_train_noisy.max():.3f}]')

# ── Visualise noisy vs clean ─────────────────────────────────────
n = 8
fig, axes = plt.subplots(2, n, figsize=(14, 3.5))
for i in range(n):
    axes[0, i].imshow(x_train_noisy[i, ..., 0], cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(x_train[i, ..., 0],       cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Noisy', fontsize=12)
axes[1, 0].set_ylabel('Clean', fontsize=12)
plt.suptitle(f'Noisy (top) vs Clean (bottom) — noise_factor={NOISE_FACTOR}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Build the Convolutional Autoencoder

### Architecture Details

| Part | Layer | Filters | Kernel | Activation |
|------|-------|---------|--------|------------|
| Encoder | Conv2D | 32 | 3×3 | ReLU |
| Encoder | MaxPool | — | 2×2 | — |
| Encoder | Conv2D | 64 | 3×3 | ReLU |
| Encoder | MaxPool | — | 2×2 | — |
| Decoder | Conv2DTranspose | 64 | 3×3 | ReLU |
| Decoder | Conv2DTranspose | 32 | 3×3 | ReLU |
| Output | Conv2D | 1 | 3×3 | Sigmoid |

In [ ]:
def build_denoising_autoencoder(input_shape=(28, 28, 1)):
    """Convolutional autoencoder for image denoising."""
    inputs = keras.Input(shape=input_shape, name='noisy_input')

    # ── Encoder ──────────────────────────────────────────────────
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='enc_conv1')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2), padding='same', name='enc_pool1')(x)   # → 14×14×32

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='enc_conv2')(x)
    x = layers.BatchNormalization()(x)
    encoded = layers.MaxPooling2D((2, 2), padding='same', name='latent')(x) # → 7×7×64

    # ── Decoder ──────────────────────────────────────────────────
    x = layers.Conv2DTranspose(64, (3, 3), strides=2, activation='relu',
                               padding='same', name='dec_conv1')(encoded)   # → 14×14×64
    x = layers.BatchNormalization()(x)

    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu',
                               padding='same', name='dec_conv2')(x)         # → 28×28×32
    x = layers.BatchNormalization()(x)

    outputs = layers.Conv2D(1, (3, 3), activation='sigmoid',
                            padding='same', name='output')(x)               # → 28×28×1

    model = Model(inputs, outputs, name='DenoisingAutoencoder')
    return model


autoencoder = build_denoising_autoencoder(IMG_SHAPE)
autoencoder.summary()

## 5. Compile and Train

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
BATCH_SIZE  = 128
EPOCHS      = 50     # Will stop early if validation loss plateaus
LEARNING_RATE = 1e-3

autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',          # Mean Squared Error between predicted and clean image
    metrics=['mae']      # Also track Mean Absolute Error
)

# ── Callbacks ────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_autoencoder.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

print('Starting training…')
history = autoencoder.fit(
    x_train_noisy, x_train,   # Input: noisy | Target: clean
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_noisy, x_test),
    callbacks=callbacks,
    verbose=1
)
print('Training complete!')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss
ax1.plot(history.history['loss'],     label='Train Loss',      color='steelblue',  lw=2)
ax1.plot(history.history['val_loss'], label='Val Loss',        color='darkorange',  lw=2, ls='--')
ax1.set_xlabel('Epoch');  ax1.set_ylabel('MSE Loss')
ax1.set_title('Training / Validation Loss')
ax1.legend();  ax1.grid(True, alpha=0.4)

# MAE
ax2.plot(history.history['mae'],     label='Train MAE',  color='steelblue',  lw=2)
ax2.plot(history.history['val_mae'], label='Val MAE',    color='darkorange',  lw=2, ls='--')
ax2.set_xlabel('Epoch');  ax2.set_ylabel('MAE')
ax2.set_title('Training / Validation MAE')
ax2.legend();  ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 7. Evaluate on Test Set

In [ ]:
test_loss, test_mae = autoencoder.evaluate(x_test_noisy, x_test, verbose=0)
print(f'Test MSE : {test_loss:.6f}')
print(f'Test MAE : {test_mae:.6f}')
print(f'Test PSNR: {10 * np.log10(1.0 / test_loss):.2f} dB')

## 8. Visual Results — Noisy vs Denoised vs Clean

In [ ]:
# Run inference on the test set
x_denoised = autoencoder.predict(x_test_noisy, verbose=0)

n = 10
indices = random.sample(range(len(x_test)), n)

fig, axes = plt.subplots(3, n, figsize=(18, 5))
row_labels = ['Noisy Input', 'Denoised Output', 'Ground Truth']
row_data   = [x_test_noisy, x_denoised, x_test]

for row, (label, data) in enumerate(zip(row_labels, row_data)):
    for col, idx in enumerate(indices):
        axes[row, col].imshow(data[idx, ..., 0], cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=40, va='center')

plt.suptitle(f'Denoising Results  (noise_factor={NOISE_FACTOR})', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('denoising_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved denoising_results.png')

## 9. Per-Image PSNR Distribution

In [ ]:
def psnr(clean, pred, max_val=1.0):
    mse = np.mean((clean - pred) ** 2, axis=(1, 2, 3))
    return 10 * np.log10(max_val**2 / (mse + 1e-10))

psnr_noisy    = psnr(x_test, x_test_noisy)
psnr_denoised = psnr(x_test, x_denoised)

print(f'Mean PSNR (noisy)    : {psnr_noisy.mean():.2f} dB')
print(f'Mean PSNR (denoised) : {psnr_denoised.mean():.2f} dB')
print(f'Improvement          : {psnr_denoised.mean() - psnr_noisy.mean():.2f} dB')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(psnr_noisy,    bins=50, alpha=0.6, label='Noisy',    color='tomato')
ax.hist(psnr_denoised, bins=50, alpha=0.6, label='Denoised', color='steelblue')
ax.axvline(psnr_noisy.mean(),    color='tomato',     ls='--', lw=2, label=f'Mean noisy={psnr_noisy.mean():.1f} dB')
ax.axvline(psnr_denoised.mean(), color='steelblue',  ls='--', lw=2, label=f'Mean denoised={psnr_denoised.mean():.1f} dB')
ax.set_xlabel('PSNR (dB)'); ax.set_ylabel('Count')
ax.set_title('PSNR Distribution: Noisy vs Denoised')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('psnr_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Visualise the Latent Space (Encoder Output)

We extract the bottleneck representation to see what compressed information the encoder preserves.

In [ ]:
# Build an encoder-only model
encoder = Model(
    inputs  = autoencoder.input,
    outputs = autoencoder.get_layer('latent').output,
    name    = 'encoder'
)

# Encode a small batch
sample_clean = x_test[:8]
sample_noisy = x_test_noisy[:8]
latent_clean = encoder.predict(sample_clean, verbose=0)  # (8, 7, 7, 64)
latent_noisy = encoder.predict(sample_noisy, verbose=0)

print(f'Latent shape: {latent_clean.shape}  →  compressed by {28*28 / (7*7):.0f}×')

# Visualise first 8 feature maps for one image
img_idx = 0
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(latent_clean[img_idx, :, :, i], cmap='viridis')
    axes[0, i].set_title(f'FM {i}', fontsize=8); axes[0, i].axis('off')
    axes[1, i].imshow(latent_noisy[img_idx, :, :, i], cmap='viridis')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Clean latent',  fontsize=10)
axes[1, 0].set_ylabel('Noisy latent',  fontsize=10)
plt.suptitle('Latent Feature Maps (first 8 of 64 channels, 7×7 grid)', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Save the Model

In [ ]:
# Save full model
autoencoder.save('denoising_autoencoder_final.keras')
print('Model saved to denoising_autoencoder_final.keras')

# Save weights only
autoencoder.save_weights('denoising_autoencoder_weights.weights.h5')
print('Weights saved to denoising_autoencoder_weights.weights.h5')

## 12. Load Model & Run Inference on a Custom Image

Use this cell to denoise your own 28×28 grayscale image.

In [ ]:
# ── Example: reload model and denoise one image ──────────────────
loaded_model = keras.models.load_model('denoising_autoencoder_final.keras')
print('Model reloaded.')

# Pick a random test image and add noise
idx          = random.randint(0, len(x_test) - 1)
clean_img    = x_test[idx]                      # (28, 28, 1)
noisy_img    = add_noise(clean_img[np.newaxis])  # (1, 28, 28, 1)
denoised_img = loaded_model.predict(noisy_img, verbose=0)  # (1, 28, 28, 1)

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, title, img in zip(
    axes,
    ['Noisy Input', 'Denoised Output', 'Ground Truth'],
    [noisy_img[0, ..., 0], denoised_img[0, ..., 0], clean_img[..., 0]]
):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Single Image Denoising Demo', fontsize=13)
plt.tight_layout()
plt.show()

## 13. (Optional) Experiment with Different Noise Levels

In [ ]:
noise_levels = [0.1, 0.2, 0.4, 0.6, 0.8]
sample_clean = x_test[:5]

fig, axes = plt.subplots(len(noise_levels) + 1, 5, figsize=(10, 10))

# Row 0: clean originals
for col in range(5):
    axes[0, col].imshow(sample_clean[col, ..., 0], cmap='gray')
    axes[0, col].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10)

# Rows 1+: denoised at different noise levels
for row, nf in enumerate(noise_levels, start=1):
    noisy = add_noise(sample_clean, noise_factor=nf)
    pred  = autoencoder.predict(noisy, verbose=0)
    for col in range(5):
        axes[row, col].imshow(pred[col, ..., 0], cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'NF={nf}', fontsize=10)

plt.suptitle('Denoised Output at Different Noise Factors', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

| Metric | Value |
|--------|-------|
| Architecture | Convolutional Autoencoder |
| Input noise factor | 0.4 (Gaussian) |
| Encoder output | 7 × 7 × 64 (3136 values) |
| Loss function | MSE |
| Optimizer | Adam |

### Tips to Improve Further
- **Increase depth**: Add more Conv2D layers in the encoder/decoder
- **Skip connections**: Use a U-Net style architecture for better detail preservation
- **Different noise types**: Try salt-and-pepper, Poisson, or speckle noise
- **Perceptual loss**: Combine MSE with a perceptual loss via a VGG feature extractor
- **Data augmentation**: Apply random rotations/flips during training